In [0]:
# ============================================================
# 00.1 BIBLIOTECAS DO FRAMEWORK
# ============================================================

# O QUE FAZ:
# Centraliza as bibliotecas utilizadas pelos notebooks do Data Quality Framework.

# COMO FAZ:
# Importa as funções e classes PySpark utilizadas pelos Profiles,Data Preparation e demais componentes do framework.

# POR QUE É IMPORTANTE:
# Evita a duplicação de imports em diferentes notebooks e estabelece um ponto único de manutenção das dependências técnicas do framework.

# PERGUNTA RESPONDIDA:
# "Quais bibliotecas são utilizadas pelo framework?"

from pyspark.sql import functions as F
from pyspark.sql.types import (
    NumericType,
    StringType,
    DateType,
    TimestampType,
    BooleanType
)

from pyspark.sql import Window

In [0]:
# ============================================================
# FUNÇÃO — NORMALIZAÇÃO DE VALORES
# ============================================================

# O QUE FAZ:
# Converte os valores das colunas para uma representação textual adequada à análise de padrões.

# COMO FAZ:
# Converte o valor para string e remove espaços nas extremidades.

# POR QUE É IMPORTANTE:
# Permite aplicar as mesmas regras de Pattern Profile sobre diferentes tipos de dados.

# PERGUNTA RESPONDIDA:
# "Como representar diferentes tipos de valores em uma estrutura comum para análise de padrões?"

def normalize_pattern_value(column):
    return F.trim(F.col(column).cast("string"))

In [0]:
# ============================================================
# FUNÇÃO — COMPRIMENTO DO VALOR
# ============================================================

# O QUE FAZ:
# Calcula o comprimento dos valores utilizados na análise de padrões.

# COMO FAZ:
# Converte o valor para string, remove espaços externos e calcula a quantidade de caracteres.

# POR QUE É IMPORTANTE:
# O comprimento é uma das primeiras características estruturais utilizadas para identificar padrões e possíveis desvios.

# PERGUNTA RESPONDIDA:
# "Qual é o comprimento dos valores presentes na coluna?"

def add_pattern_length(column):
    valor = normalize_pattern_value(column)
    return F.length(valor)

In [0]:
# ============================================================
# FUNÇÃO — COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Classifica cada valor de acordo com sua composição de caracteres.

# COMO FAZ:
# Utiliza expressões regulares para identificar valores:
#
# NUMÉRICOS
# ALFABÉTICOS
# ALFANUMÉRICOS
# SÍMBOLOS
# MISTOS

# POR QUE É IMPORTANTE:
# Permite identificar alterações estruturais mesmo quando o valor possui o mesmo significado aparente.

# PERGUNTA RESPONDIDA:
# "Qual é a composição estrutural dos valores da coluna?"

def classify_character_composition(column):
    value = normalize_pattern_value(column)

    return (
        F.when(value.isNull() | (value == ""), "EMPTY")
        .when(value.rlike("^[0-9]+$"), "NUMERIC")
        .when(value.rlike(r"^[A-Za-zÀ-ÿ]+$"), "ALPHABETIC")
        .when( 
            value.rlike(r"^[A-Za-zÀ-ÿ0-9]+$") &
            value.rlike(r"[A-Za-zÀ-ÿ]") &
            value.rlike(r"[0-9]"), 
            "ALPHANUMERIC"
        )
        .when(value.rlike(r"^[^A-Za-zÀ-ÿ0-9]+$"), "SYMBOLS")
        .otherwise("MIXED")

    )

In [0]:
# ============================================================
# FUNÇÃO — ESTRUTURA DO VALOR
# ============================================================

# O QUE FAZ:
# Converte cada caractere do valor em uma representação estrutural padronizada.

# COMO FAZ:
# Classifica:
#
# Letras  -> L
# Números -> N
# Espaços -> S
# Outros  -> X
#
# Exemplo:
#
# ABC123      -> LLLNNN
# 123456      -> NNNNNN
# AB-123      -> LLXNNN
# 12/05/2026  -> NN X NN X NNNN

# POR QUE É IMPORTANTE:
# Permite identificar padrões estruturais sem depender do significado semântico ou do nome da coluna.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura dos valores presentes na coluna?"

def generate_pattern_structure(column):

    value = normalize_pattern_value(column)

    return (
        F.when(
            value.isNull() | (value == ""),
            "VAZIO"
        )
        .otherwise(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(
                            value,
                            r"[A-Za-zÀ-ÿ]",
                            "L"
                        ),
                        r"[0-9]",
                        "N"
                    ),
                    r"\s",
                    "S"
                ),
                r"[^A-Za-zÀ-ÿ0-9\s]",
                "X"
            )
        )
    )

In [0]:
# ============================================================
# FUNÇÃO — RESUMO DOS PADRÕES
# ============================================================

# O QUE FAZ:
# Calcula a estrutura predominante de cada coluna e sua respectiva participação percentual.

# COMO FAZ:
# Conta a frequência de cada estrutura encontrada e utiliza uma janela para identificar a estrutura mais frequente.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente qual estrutura domina uma determinada coluna.

# PERGUNTA RESPONDIDA:
# "Qual é o padrão estrutural predominante de cada coluna?"

def generate_pattern_summary(df, columns, total_records):
    results = []

    for column in columns:
        column_structure = (
            df
            .select(F.lit(column).alias("column"),
                generate_pattern_structure(
                    F.col(column)
                ).alias("structure")

            )
            .groupBy(
                "column",
                "structure"
            )
            .agg(
                F.count("*").alias("frequency"
            )
                 
            )
            .withColumn(
                "pct_structure",
                F.col("frequency") / F.lit(total_records) * 100
            )
        )
    results.append(column_structure)
    result = results[0]

    for df_results in results[1:]:
        result = result.unionByName(df_results)

    window_pattern = (
        Window
        .partitionBy("column")
        .orderBy(
            F.desc("frequency"),
            F.asc("pct_structure")
        )
    )

    return (
        result
        .withColumn(
            "ranking",
            F.row_number().over(window_pattern)
        )
    )

In [0]:
# ============================================================
# FUNÇÃO — RESUMO DA COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Calcula a distribuição das categorias de composição dos valores em cada coluna.

# COMO FAZ:
# Classifica cada valor e calcula frequência e percentual de cada categoria estrutural.

# POR QUE É IMPORTANTE:
# Permite identificar mudanças na composição dos dados, como mistura inesperada entre números, letras e símbolos.

# PERGUNTA RESPONDIDA:
# "Qual composição de caracteres predomina em cada coluna?"

def generate_character_composition(df, columns, total_records):
    results = []

    for column in columns:
        composition = (
            df
            .select(
                F.lit(column).alias("column"),
                classify_character_composition(
                    F.col(column)
                )
                .groupBy("column", "composition")
                .agg(F.count("*").alias("frequency"))
                .withColumn("pct_composition", F.col("frequency")/ F.lit(total_records) * 100)
            )
        )
        results.append(composition)
    result = results[0]

    for df_result in results[1:]:
        result = result.unionByName(df_result)
    return result